# 04 - Express Mode and Selective Replay

Notebook 03 built the full certification engine pipeline. In production, we want the
architect to see everything first and then go back and tweak specific artifacts.

**Express Mode** runs the full pipeline uninterrupted, then lets
the architect:
1. Review all artifacts at once
2. Pick one to edit (e.g., "the competency framework needs a 5th domain")
3. Inject the edit via `update_state()`
4. Replay the graph from that point forward - all downstream artifacts regenerate

## Key Concept: `update_state()`
The checkpointer saves state after every node. `update_state()` lets you manually modify
the state at a specific checkpoint and then re-invoke the graph. Downstream nodes see the
edited state and regenerate accordingly. The architect edits the framework, and assessments,
rubrics, item bank, and blueprint all rebuild from the edited version.

In [1]:
import os
import json
import copy
from pathlib import Path
from typing import TypedDict, Optional
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

from IPython.display import Image, display

load_dotenv(Path("../.env"))

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME = "certops_docs"

print("Imports loaded.")

Imports loaded.


## 1. Schemas, State, and Node Functions (No Interrupts)

Same schemas and state as notebooks 03-04. The node functions here are the **original
versions without `interrupt()`** - the pipeline runs straight through.

In [2]:
# ── Schemas ──

class Skill(BaseModel):
    name: str = Field(description="Name of the skill")
    description: str = Field(description="What this skill covers")
    proficiency_levels: list[str] = Field(
        description="Proficiency descriptors: novice, competent, expert",
        min_length=3, max_length=3,
    )

class Domain(BaseModel):
    name: str = Field(description="Domain name")
    description: str = Field(description="What this domain covers")
    skills: list[Skill] = Field(description="Skills within this domain", min_length=2)

class CompetencyFramework(BaseModel):
    track: str = Field(description="Certification track name")
    description: str = Field(description="Overview of what this certification validates")
    domains: list[Domain] = Field(description="Competency domains", min_length=3)

class LearningObjective(BaseModel):
    order: int = Field(description="Sequence order (1-based)")
    title: str = Field(description="Learning objective title")
    description: str = Field(description="What the learner will be able to do")
    domain: str = Field(description="Which competency domain this maps to")
    prerequisites: list[str] = Field(default_factory=list)

class LearningProgression(BaseModel):
    track: str = Field(description="Certification track name")
    objectives: list[LearningObjective] = Field(description="Ordered learning objectives")

class AssessmentTask(BaseModel):
    title: str = Field(description="Assessment task title")
    scenario: str = Field(description="Real-world scenario")
    instructions: str = Field(description="Detailed instructions")
    expected_outputs: list[str] = Field(description="What the candidate should produce")
    competency_ref: str = Field(description="Which skill/domain this assesses")
    time_estimate_minutes: int = Field(description="Estimated time to complete")

class AssessmentList(BaseModel):
    tasks: list[AssessmentTask] = Field(description="List of assessment tasks")

class RubricCriterion(BaseModel):
    criterion: str = Field(description="What is being evaluated")
    novice: str = Field(description="Novice-level descriptor")
    competent: str = Field(description="Competent-level descriptor")
    expert: str = Field(description="Expert-level descriptor")

class Rubric(BaseModel):
    assessment_ref: str = Field(description="Title of the assessment task")
    criteria: list[RubricCriterion] = Field(description="Evaluation criteria", min_length=3)

class RubricList(BaseModel):
    rubrics: list[Rubric] = Field(description="List of rubrics")

class ItemBankEntry(BaseModel):
    stem: str = Field(description="The question or task prompt")
    task_type: str = Field(description="performance, scenario, or analysis")
    competency_ref: str = Field(description="Skill/domain this item assesses")
    expected_response_summary: str = Field(description="Brief summary of correct response")
    scoring_notes: str = Field(description="Notes for evaluators")

class ItemBank(BaseModel):
    items: list[ItemBankEntry] = Field(description="Item bank entries")

class CertificationBlueprint(BaseModel):
    program_title: str = Field(description="Full certification program title")
    target_audience: str = Field(description="Who this certification is for")
    prerequisites: str = Field(description="What candidates should know before starting")
    program_overview: str = Field(description="2-3 paragraph program overview")
    domain_summary: list[str] = Field(description="One sentence summary per domain")
    assessment_strategy: str = Field(
        description="How candidates are assessed - format, approach, and philosophy"
    )
    estimated_duration_hours: float = Field(
        description="Total estimated hours to complete the certification"
    )
    renewal_cadence: str = Field(
        description="Recommended recertification interval and rationale"
    )

print("Schemas defined.")

Schemas defined.


In [3]:
# ── State, helpers, and node functions (no interrupt) ──

class CertOpsState(TypedDict):
    track: str
    documents: list[str]
    tavily_context: str
    competency_framework: Optional[dict]
    learning_progression: Optional[dict]
    assessments: Optional[list[dict]]
    rubrics: Optional[list[dict]]
    item_bank: Optional[list[dict]]
    blueprint: Optional[dict]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

llm = ChatOpenAI(model="gpt-4o", temperature=0)

TRACK_SEARCH_QUERIES = {
    "AI Champion": "Microsoft Copilot Studio latest updates 2026",
    "M365 Copilot User": "Microsoft 365 Copilot latest updates 2026",
}

TRACK_DOMAIN_HINTS = {
    "AI Champion": (
        "Include at least 4 domains covering agent creation & configuration, "
        "conversational design, connectors/integrations, and security/governance."
    ),
    "M365 Copilot User": (
        "Include at least 4 domains covering productivity (Word, Excel, PowerPoint), "
        "communication (Teams, Outlook), data analysis, and prompting best practices."
    ),
}


def retrieve_docs(state: CertOpsState) -> dict:
    track = state["track"]
    query = f"{track} certification competencies and skills"
    wide_retriever = vector_store.as_retriever(search_kwargs={"k": 20})
    compressor = CohereRerank(model="rerank-v3.5", top_n=5)
    reranked_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=wide_retriever,
    )
    docs = reranked_retriever.invoke(query)
    doc_texts = [doc.page_content for doc in docs]

    tavily = TavilySearch(max_results=3)
    try:
        search_query = TRACK_SEARCH_QUERIES.get(track, f"{track} latest updates 2026")
        tavily_results = tavily.invoke(search_query)
        tavily_context = str(tavily_results)[:2000]
    except Exception:
        tavily_context = ""
    return {"documents": doc_texts, "tavily_context": tavily_context}


def check_documents(state: CertOpsState) -> str:
    if len(state.get("documents", [])) < 3:
        return "retry_retrieve"
    return "generate"


def generate_competency_framework(state: CertOpsState) -> dict:
    structured_llm = llm.with_structured_output(CompetencyFramework)
    context = "\n\n".join(state["documents"])
    domain_hint = TRACK_DOMAIN_HINTS.get(state["track"], "Include at least 4 relevant domains.")
    response = structured_llm.invoke([
        SystemMessage(content=(
            "You are an expert certification architect specializing in enterprise AI platforms. "
            "Using the provided documentation context, generate a comprehensive competency framework "
            f"for the specified track. {domain_hint}\n\n"
            f"Context:\n{context}\n\nLatest updates:\n{state.get('tavily_context', '')}"
        )),
        HumanMessage(content=f"Generate a competency framework for: {state['track']}"),
    ])
    return {"competency_framework": response.model_dump()}


def generate_learning_progression(state: CertOpsState) -> dict:
    structured_llm = llm.with_structured_output(LearningProgression)
    fw_str = json.dumps(state["competency_framework"], indent=2)
    response = structured_llm.invoke([
        SystemMessage(content=(
            "You are an instructional designer. Create an ordered learning progression "
            "from this framework. Earlier objectives are prerequisites for later ones."
        )),
        HumanMessage(content=f"Create learning progression for:\n{fw_str}"),
    ])
    return {"learning_progression": response.model_dump()}


def generate_assessments(state: CertOpsState) -> dict:
    structured_llm = llm.with_structured_output(AssessmentList)
    fw_str = json.dumps(state["competency_framework"], indent=2)
    context = "\n\n".join(state["documents"][:5])
    response = structured_llm.invoke([
        SystemMessage(content=(
            "You are an assessment designer. Generate performance-based tasks that evaluate "
            f"real competence, not just recall.\n\nFramework:\n{fw_str}\n\nContext:\n{context}"
        )),
        HumanMessage(content="Generate one assessment task per domain."),
    ])
    return {"assessments": [t.model_dump() for t in response.tasks]}


def generate_rubrics(state: CertOpsState) -> dict:
    structured_llm = llm.with_structured_output(RubricList)
    assessments_str = json.dumps(state["assessments"], indent=2)
    response = structured_llm.invoke([
        SystemMessage(content=(
            "You are an expert in rubric design. Create scoring rubrics with "
            "novice/competent/expert descriptors for consistent inter-rater reliability."
        )),
        HumanMessage(content=f"Create rubrics for:\n{assessments_str}"),
    ])
    return {"rubrics": [r.model_dump() for r in response.rubrics]}


def generate_item_bank(state: CertOpsState) -> dict:
    structured_llm = llm.with_structured_output(ItemBank)
    fw_str = json.dumps(state["competency_framework"], indent=2)
    context = "\n\n".join(state["documents"][:5])
    response = structured_llm.invoke([
        SystemMessage(content=(
            "You are an expert item writer. Generate performance/scenario/analysis items "
            f"(NOT multiple choice).\n\nFramework:\n{fw_str}\n\nContext:\n{context}"
        )),
        HumanMessage(content="Generate 10 item bank entries spanning all domains."),
    ])
    return {"item_bank": [item.model_dump() for item in response.items]}


def generate_blueprint(state: CertOpsState) -> dict:
    structured_llm = llm.with_structured_output(CertificationBlueprint)
    summary = {
        "track": state["track"],
        "framework": state["competency_framework"],
        "num_objectives": len(state["learning_progression"]["objectives"]),
        "num_assessments": len(state["assessments"]),
        "num_rubrics": len(state["rubrics"]),
        "num_items": len(state["item_bank"]),
    }
    response = structured_llm.invoke([
        SystemMessage(content=(
            "You are a certification program director writing an executive summary. "
            "Synthesize all artifacts into a cohesive Certification Blueprint.\n\n"
            "The program_overview should be 2-3 substantive paragraphs. "
            "The assessment_strategy should explain the performance-based philosophy."
        )),
        HumanMessage(content=f"Create a certification blueprint from:\n{json.dumps(summary, indent=2)}"),
    ])
    return {"blueprint": response.model_dump()}


print("All node functions defined (Express Mode - no interrupts).")

All node functions defined (Express Mode - no interrupts).


## 2. Build the Express Mode Graph

Same graph structure as always, but compiled with a checkpointer and **no interrupts**.
The pipeline runs all 7 nodes straight through. The checkpointer silently saves state
after every node - that's what makes selective replay possible later.

In [4]:
checkpointer = MemorySaver()

builder = StateGraph(CertOpsState)

builder.add_node("retrieve_docs", retrieve_docs)
builder.add_node("generate_competency_framework", generate_competency_framework)
builder.add_node("generate_learning_progression", generate_learning_progression)
builder.add_node("generate_assessments", generate_assessments)
builder.add_node("generate_rubrics", generate_rubrics)
builder.add_node("generate_item_bank", generate_item_bank)
builder.add_node("generate_blueprint", generate_blueprint)

builder.add_edge(START, "retrieve_docs")
builder.add_conditional_edges(
    "retrieve_docs",
    check_documents,
    {"retry_retrieve": "retrieve_docs", "generate": "generate_competency_framework"},
)
builder.add_edge("generate_competency_framework", "generate_learning_progression")
builder.add_edge("generate_learning_progression", "generate_assessments")
builder.add_edge("generate_assessments", "generate_rubrics")
builder.add_edge("generate_rubrics", "generate_item_bank")
builder.add_edge("generate_item_bank", "generate_blueprint")
builder.add_edge("generate_blueprint", END)

graph = builder.compile(checkpointer=checkpointer)
print("Express Mode graph compiled with checkpointer (no interrupts).")

Express Mode graph compiled with checkpointer (no interrupts).


## 3. Run the Full Pipeline (Express Mode)

One invoke, all 7 nodes, no pauses. The architect sees everything at once.

In [5]:
thread_config = {"configurable": {"thread_id": "express-demo-1"}}

initial_state: CertOpsState = {
    "track": "AI Champion",
    "documents": [],
    "tavily_context": "",
    "competency_framework": None,
    "learning_progression": None,
    "assessments": None,
    "rubrics": None,
    "item_bank": None,
    "blueprint": None,
}

print("Running full pipeline (Express Mode)...")
print("No pauses - all 7 nodes run in sequence.\n")

result = graph.invoke(initial_state, config=thread_config)

print("Pipeline complete!\n")
print(f"  Framework domains: {len(result['competency_framework']['domains'])}")
print(f"  Learning objectives: {len(result['learning_progression']['objectives'])}")
print(f"  Assessments: {len(result['assessments'])}")
print(f"  Rubrics: {len(result['rubrics'])}")
print(f"  Item bank entries: {len(result['item_bank'])}")
print(f"  Blueprint: {result['blueprint']['program_title']}")

Running full pipeline (Express Mode)...
No pauses - all 7 nodes run in sequence.



/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CompetencyFramework(track...mpetent', 'expert'])])]), input_type=CompetencyFramework])
  return self.__pydantic_serializer__.to_python(
/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=LearningProgression(track...overnance Adherence'])]), input_type=LearningProgression])
  return self.__pydantic_serializer__.to_python(
/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializa

Pipeline complete!

  Framework domains: 4
  Learning objectives: 12
  Assessments: 4
  Rubrics: 4
  Item bank entries: 10
  Blueprint: AI Champion Certification


/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CertificationBlueprint(pr... continues to advance.'), input_type=CertificationBlueprint])
  return self.__pydantic_serializer__.to_python(


In [6]:
# Save the original output for comparison later
original_result = copy.deepcopy(result)

print("COMPETENCY FRAMEWORK")
print("=" * 60)
for domain in result["competency_framework"]["domains"]:
    skills = ", ".join(s["name"] for s in domain["skills"])
    print(f"  {domain['name']}: {skills}")

print(f"\nASSESSMENTS ({len(result['assessments'])})")
print("=" * 60)
for t in result["assessments"]:
    print(f"  {t['title']} ({t['time_estimate_minutes']}min) - {t['competency_ref']}")

print(f"\nITEM BANK ({len(result['item_bank'])} items)")
print("=" * 60)
for item in result["item_bank"][:3]:
    print(f"  [{item['task_type']}] {item['stem'][:80]}...")

COMPETENCY FRAMEWORK
  Agent Creation & Configuration: Agent Configuration, Action Definition, Trigger Management
  Conversational Design: Topic Management, Evaluation and Testing, User Engagement Analysis
  Connectors/Integrations: Channel Deployment, System Integration, API Utilization
  Security & Governance: Security Configuration, Governance Adherence, Performance Analytics

ASSESSMENTS (4)
  Configure an AI Agent for Customer Support (120min) - Agent Creation & Configuration
  Design a Conversational Flow for a Banking Chatbot (150min) - Conversational Design
  Integrate an AI Agent with a CRM System (180min) - Connectors/Integrations
  Implement Security and Governance for an AI Agent (120min) - Security & Governance

ITEM BANK (10 items)
  [scenario] Design a conversation flow for an AI agent that handles customer inquiries about...
  [analysis] Analyze the user engagement data from the Analytics tab to identify areas where ...
  [performance] Configure an AI agent to authentic

## 4. Walk the Checkpoint History

The pipeline ran all 7 nodes, and the checkpointer saved state after each one.
Let's look at the full history to understand what's available for replay.

Each checkpoint records which node just completed and what state looked like at that point.
This is the map we use to figure out *where* to inject an edit.

In [7]:
history = list(graph.get_state_history(thread_config))

print(f"Total checkpoints: {len(history)}\n")
print(f"{'#':<4} {'Next Node':<40} {'Has Framework':<16} {'Has Blueprint'}")
print("=" * 80)
for i, cp in enumerate(history):
    next_nodes = cp.next if cp.next else ("(completed)",)
    has_fw = cp.values.get("competency_framework") is not None
    has_bp = cp.values.get("blueprint") is not None
    print(f"{i:<4} {str(next_nodes):<40} {str(has_fw):<16} {has_bp}")

Total checkpoints: 9

#    Next Node                                Has Framework    Has Blueprint
0    ('(completed)',)                         True             True
1    ('generate_blueprint',)                  True             False
2    ('generate_item_bank',)                  True             False
3    ('generate_rubrics',)                    True             False
4    ('generate_assessments',)                True             False
5    ('generate_learning_progression',)       True             False
6    ('generate_competency_framework',)       False            False
7    ('retrieve_docs',)                       False            False
8    ('__start__',)                           False            False


## 5. Edit and Replay

The architect reviews the output and decides: "The framework needs a 5th domain for
Responsible AI." Instead of regenerating everything from scratch, we:

1. **Edit** the competency framework to add the new domain
2. **Inject** the edit via `update_state()` at the point right after the framework was generated
3. **Re-invoke** the graph - it picks up from the framework checkpoint and regenerates
   learning progression, assessments, rubrics, item bank, and blueprint

Only the downstream nodes re-run. Retrieval doesn't repeat. The framework uses your edit.

In [8]:
# Step 1: Edit the framework - add a 5th domain for Responsible AI
edited_framework = copy.deepcopy(original_result["competency_framework"])

new_domain = {
    "name": "Responsible AI Governance",
    "description": "Ensuring AI agents adhere to fairness, transparency, accountability, and safety standards",
    "skills": [
        {
            "name": "Content Safety Configuration",
            "description": "Configuring content moderation filters and safety guardrails in Copilot Studio agents",
            "proficiency_levels": [
                "Aware of content safety concepts but relies on default settings",
                "Configures custom content filters, sets up moderation rules, and tests edge cases",
                "Designs organization-wide content safety policies, audits agent behavior, and implements advanced safety measures",
            ],
        },
        {
            "name": "AI Ethics and Compliance",
            "description": "Applying responsible AI principles to agent design, deployment, and monitoring",
            "proficiency_levels": [
                "Can identify basic responsible AI principles but cannot apply them to agent configuration",
                "Integrates fairness checks, documents AI decision-making processes, and follows compliance guidelines",
                "Leads responsible AI governance programs, conducts impact assessments, and establishes organizational AI ethics frameworks",
            ],
        },
    ],
}

edited_framework["domains"].append(new_domain)

print(f"Original: {len(original_result['competency_framework']['domains'])} domains")
print(f"Edited:   {len(edited_framework['domains'])} domains")
print(f"\nAdded domain: '{new_domain['name']}'")
print(f"  Skills: {', '.join(s['name'] for s in new_domain['skills'])}")

Original: 4 domains
Edited:   5 domains

Added domain: 'Responsible AI Governance'
  Skills: Content Safety Configuration, AI Ethics and Compliance


In [9]:
# Step 2: Inject the edit via update_state
# as_node tells the checkpointer "pretend this update came from generate_competency_framework"
# so the graph knows to resume from the NEXT node (generate_learning_progression)

graph.update_state(
    thread_config,
    values={"competency_framework": edited_framework},
    as_node="generate_competency_framework",
)

# Verify the state was updated
updated_state = graph.get_state(thread_config)
print(f"State updated. Next node to run: {updated_state.next}")
print(f"Framework now has {len(updated_state.values['competency_framework']['domains'])} domains")

State updated. Next node to run: ('generate_learning_progression',)
Framework now has 5 domains


In [10]:
# Step 3: Re-invoke the graph - it replays from the framework checkpoint forward
print("Replaying pipeline from competency framework forward...")
print("Nodes that will re-run: learning progression, assessments, rubrics, item bank, blueprint\n")

edited_result = graph.invoke(None, config=thread_config)

print("Replay complete!\n")
print(f"  Framework domains: {len(edited_result['competency_framework']['domains'])}")
print(f"  Learning objectives: {len(edited_result['learning_progression']['objectives'])}")
print(f"  Assessments: {len(edited_result['assessments'])}")
print(f"  Rubrics: {len(edited_result['rubrics'])}")
print(f"  Item bank entries: {len(edited_result['item_bank'])}")
print(f"  Blueprint: {edited_result['blueprint']['program_title']}")

Replaying pipeline from competency framework forward...
Nodes that will re-run: learning progression, assessments, rubrics, item bank, blueprint



/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=LearningProgression(track...afety Configuration'])]), input_type=LearningProgression])
  return self.__pydantic_serializer__.to_python(
/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AssessmentList(tasks=[Ass..._estimate_minutes=180)]), input_type=AssessmentList])
  return self.__pydantic_serializer__.to_python(
/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationU

Replay complete!

  Framework domains: 5
  Learning objectives: 14
  Assessments: 5
  Rubrics: 5
  Item bank entries: 10
  Blueprint: AI Champion Certification


/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=CertificationBlueprint(pr... of the certification.'), input_type=CertificationBlueprint])
  return self.__pydantic_serializer__.to_python(


## 6. Compare Original vs Edited

The whole point of selective replay: downstream artifacts should now reflect the new
Responsible AI Governance domain. Let's compare.

In [11]:
print("FRAMEWORK COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<25} {'Original':>12} {'Edited':>12}")
print("-" * 50)
print(f"{'Domains':<25} {len(original_result['competency_framework']['domains']):>12} {len(edited_result['competency_framework']['domains']):>12}")
print(f"{'Learning objectives':<25} {len(original_result['learning_progression']['objectives']):>12} {len(edited_result['learning_progression']['objectives']):>12}")
print(f"{'Assessment tasks':<25} {len(original_result['assessments']):>12} {len(edited_result['assessments']):>12}")
print(f"{'Rubrics':<25} {len(original_result['rubrics']):>12} {len(edited_result['rubrics']):>12}")
print(f"{'Item bank entries':<25} {len(original_result['item_bank']):>12} {len(edited_result['item_bank']):>12}")

print("\n\nORIGINAL DOMAINS:")
for d in original_result["competency_framework"]["domains"]:
    print(f"  {d['name']}")

print("\nEDITED DOMAINS:")
for d in edited_result["competency_framework"]["domains"]:
    print(f"  {d['name']}")

FRAMEWORK COMPARISON

Metric                        Original       Edited
--------------------------------------------------
Domains                              4            5
Learning objectives                 12           14
Assessment tasks                     4            5
Rubrics                              4            5
Item bank entries                   10           10


ORIGINAL DOMAINS:
  Agent Creation & Configuration
  Conversational Design
  Connectors/Integrations
  Security & Governance

EDITED DOMAINS:
  Agent Creation & Configuration
  Conversational Design
  Connectors/Integrations
  Security & Governance
  Responsible AI Governance


In [12]:
# Check if the new domain shows up in downstream artifacts
print("RESPONSIBLE AI IN DOWNSTREAM ARTIFACTS")
print("=" * 60)

# Check assessments
rai_assessments = [a for a in edited_result["assessments"]
                   if "responsible" in a.get("competency_ref", "").lower()
                   or "responsible" in a.get("title", "").lower()
                   or "safety" in a.get("title", "").lower()
                   or "ethics" in a.get("title", "").lower()
                   or "governance" in a.get("competency_ref", "").lower()]
print(f"\nAssessments referencing Responsible AI: {len(rai_assessments)}")
for a in rai_assessments:
    print(f"  - {a['title']} ({a['competency_ref']})")

# Check item bank
rai_items = [i for i in edited_result["item_bank"]
             if "responsible" in i.get("competency_ref", "").lower()
             or "safety" in i.get("competency_ref", "").lower()
             or "ethics" in i.get("competency_ref", "").lower()
             or "governance" in i.get("competency_ref", "").lower()]
print(f"\nItem bank entries referencing Responsible AI: {len(rai_items)}")
for i in rai_items:
    print(f"  - [{i['task_type']}] {i['stem'][:70]}...")

# Check blueprint domain summaries
print(f"\nBlueprint domain summaries ({len(edited_result['blueprint']['domain_summary'])}):")
for ds in edited_result["blueprint"]["domain_summary"]:
    print(f"  - {ds[:80]}...")

RESPONSIBLE AI IN DOWNSTREAM ARTIFACTS

Assessments referencing Responsible AI: 2
  - Implement Security and Governance for an AI Agent (Security & Governance)
  - Develop a Responsible AI Governance Framework (Responsible AI Governance)

Item bank entries referencing Responsible AI: 4
  - [scenario] Configure an AI agent to authenticate users before providing access to...
  - [scenario] Describe how you would configure content safety settings for an AI age...
  - [performance] Develop a governance framework for ensuring AI agents comply with resp...
  - [analysis] Analyze the performance analytics of an AI agent to ensure it complies...

Blueprint domain summaries (5):
  - Agent Creation & Configuration: Skills for creating and configuring AI agents, i...
  - Conversational Design: Designing effective conversation flows and managing topic...
  - Connectors/Integrations: Integrating AI agents with external systems and deployi...
  - Security & Governance: Ensuring AI agents adhere to s

## 7. Checkpoint History After the Edit

Let's look at what the checkpoint timeline looks like now. The edit created a *fork*:
the original run is still there, plus a new branch where the framework was updated and
downstream nodes re-ran.

In [13]:
history_after_edit = list(graph.get_state_history(thread_config))

print(f"Total checkpoints after edit: {len(history_after_edit)}")
print(f"(Before edit: {len(history)})\n")
print(f"{'#':<4} {'Checkpoint ID (last 8)':<24} {'Next Node':<40} {'Domains'}")
print("=" * 90)
for i, cp in enumerate(history_after_edit):
    next_nodes = cp.next if cp.next else ("(completed)",)
    fw = cp.values.get("competency_framework")
    n_domains = len(fw["domains"]) if fw else "-"
    cp_id = cp.config["configurable"]["checkpoint_id"][-8:]
    print(f"{i:<4} {cp_id:<24} {str(next_nodes):<40} {n_domains}")

Total checkpoints after edit: 15
(Before edit: 9)

#    Checkpoint ID (last 8)   Next Node                                Domains
0    79c96dc6                 ('(completed)',)                         5
1    26b656ca                 ('generate_blueprint',)                  5
2    2d31bf0c                 ('generate_item_bank',)                  5
3    a39c021e                 ('generate_rubrics',)                    5
4    cef88d45                 ('generate_assessments',)                5
5    ad3ccf7c                 ('generate_learning_progression',)       5
6    c604d816                 ('(completed)',)                         4
7    a4ee1b87                 ('generate_blueprint',)                  4
8    7830754c                 ('generate_item_bank',)                  4
9    da7c6d71                 ('generate_rubrics',)                    4
10   1aabbbb9                 ('generate_assessments',)                4
11   e4fa7f85                 ('generate_learning_progression',)   

## 8. Export Both Versions

Save both the original and edited outputs for the architect to compare side-by-side.

In [14]:
output_dir = Path("../data/output")
output_dir.mkdir(parents=True, exist_ok=True)

EXPORT_KEYS = [
    "competency_framework", "learning_progression",
    "assessments", "rubrics", "item_bank", "blueprint",
]

original_export = {k: original_result[k] for k in EXPORT_KEYS}
edited_export = {k: edited_result[k] for k in EXPORT_KEYS}

original_path = output_dir / "express_original.json"
edited_path = output_dir / "express_edited.json"

with open(original_path, "w") as f:
    json.dump(original_export, f, indent=2)

with open(edited_path, "w") as f:
    json.dump(edited_export, f, indent=2)

print(f"Original exported: {original_path}")
print(f"Edited exported:   {edited_path}")

Original exported: ../data/output/express_original.json
Edited exported:   ../data/output/express_edited.json


## Recap

| Concept | What It Does | Where We Used It |
|---------|-------------|-----------------|
| **Express Mode** | Full pipeline, no pauses | Cells 3-4: one `invoke()` call |
| **Checkpoint history** | See every saved state | Cell 5: `get_state_history()` |
| **`update_state()`** | Inject edits into a checkpoint | Cell 7: added 5th domain |
| **`as_node`** | Tell the graph which node "produced" the edit | Cell 7: `as_node="generate_competency_framework"` |
| **Selective replay** | Re-invoke from the edit point forward | Cell 8: only downstream nodes re-ran |
| **Fork history** | Original and edited runs coexist | Cell 10: checkpoint count grew |

### What This Means for Production

In the frontend, the Express Mode flow looks like:
1. Architect selects a track and clicks "Generate All"
2. Pipeline runs end-to-end, all artifacts appear
3. Architect reviews and clicks "Edit" on any artifact
4. Frontend sends the edit to the backend, which calls `update_state()`
5. Backend re-invokes the graph - only downstream nodes re-run
6. Updated artifacts replace the old ones in the UI

Combined with Guided Mode (Notebook 04), the architect has two complementary workflows:
- **Guided Mode** for careful, step-by-step artifact review
- **Express Mode** for quick generation with targeted edits

### Production Implementation

These patterns are now implemented in the production CertOps application — the FastAPI backend uses `update_state()` and selective replay to power the Express Mode editing workflow in the Build Your Own feature.